# Decision-effective conformal ambiguity sets for neural PDE control

This portable notebook documents the current public result and the locked next experiment. It contains no platform-specific setup. Install the project from the repository README before running it.

**Scientific status.** The persistent-forcing Burgers task-validity gate is complete. The five-seed learned-controller comparison is not yet complete, so this notebook does not claim that robust FNO MPC outperforms nominal MPC.

In [ ]:
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    matches = [p for p in PROJECT_ROOT.parents if (p / 'pyproject.toml').exists()]
    if not matches:
        raise RuntimeError('Run this notebook from the cloned repository or one of its subdirectories.')
    PROJECT_ROOT = matches[0]

RESULT_ROOT = PROJECT_ROOT / 'results' / 'forced_oracle_validation'
PROJECT_ROOT

## 1. Inspect the validated task gate

The benchmark uses persistent external forcing and two localized actuators. PDE-oracle MPC plans with the numerical solver itself. Oracle therefore means model access, not globally optimal planning.

In [ ]:
summary = json.loads((RESULT_ROOT / 'summary.json').read_text())
for name in ('uncontrolled', 'pde_oracle_mpc'):
    row = summary[name]
    print(
        f"{name:18s} mean={row['mean_cost']:.4f}  "
        f"median={row['median_cost']:.4f}  p90={row['p90_cost']:.4f}"
    )
print('paired mean difference:', summary['paired']['mean_difference'])
print('fraction oracle better:', summary['paired']['fraction_oracle_better'])

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(RESULT_ROOT / 'figures' / 'oracle_01_paired_cost.png'), width=520))

## 2. Re-run the numerical experiment

The following cell runs a small smoke experiment by default. Set RUN_FULL to True to reproduce the 100-case release. Output is written to a new project-relative directory, leaving the released results untouched.

In [ ]:
from unoc.forced_control import run_task_validation

RUN_FULL = False
cases = 100 if RUN_FULL else 5
replicates = 5000 if RUN_FULL else 200
output = PROJECT_ROOT / 'results' / ('forced_oracle_reproduction' if RUN_FULL else 'forced_oracle_smoke')

rerun = run_task_validation(
    output_dir=output,
    cases=cases,
    rollout_horizon=20,
    seed=27,
    bootstrap_replicates=replicates,
)
rerun['paired']

## 3. Locked next experiment

The next release must train five independent FNO seeds and evaluate uncontrolled, PDE-oracle, nominal, audit-L2, ellipsoid-adjoint, and box-adjoint control on shared independent joint-shift trajectories. It must also compare source calibration with target auditing, repeat audit sizes 20/50/100/200/300, and ablate the uncertainty scale.

The intended claim is deliberately narrow: at matched field-level coverage, ambiguity-set geometry changes adjoint support, robust actions, and upper-tail closed-loop cost.